# 40 — Assemble Tier 2b VLM_Narrative

*One deterministically assembled English paragraph per DSS (1,326
paragraphs). Output: `sdtm-narrative/interim/COSMoS_Graph_Narrative.xlsx`,
sheet `Narratives`, one row per DSS with the `VLM_Narrative` text.*

**Grain decision (§1 of the design record):** per-DSS paragraph.
The paragraph is the proposition about what this row-template pins,
what it leaves open, and what SDTM domain context it carries.

**Not LLM-generated.** The paragraph is a deterministic projection
of the graph rows (DSS + Variables + BC) through the template
catalogue's natural-name substitution. No free-text synthesis.

**Reuses from 60:** `var_nn()` (two-tier substitution),
`dss_attributes()` (Variables-row projection), `structural_type()`
(domain → specimen/instrument/measurement). These are lifted into
this notebook unchanged.

## Setup — paths, imports, loads

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

REPO = Path("/Users/kerstinforsberg/repos/GitHub/cdisc-for-ai")

GRAPH       = REPO / "cosmos-graph/interim/COSMoS_Graph.xlsx"
CT          = REPO / "cosmos-graph/interim/COSMoS_Graph_CT.xlsx"
VAR_IDENT   = REPO / "sdtm-narrative/reference/SDTM_Variable_Identity.xlsx"
DOMAIN_META = REPO / "sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx"

OUT_XLSX = REPO / "sdtm-narrative/interim/COSMoS_Graph_Narrative.xlsx"
OUT_XLSX.parent.mkdir(parents=True, exist_ok=True)

bc         = pd.read_excel(GRAPH, sheet_name="BC")
dss        = pd.read_excel(GRAPH, sheet_name="DSS")
variables  = pd.read_excel(GRAPH, sheet_name="Variables")

var_ident    = pd.read_excel(VAR_IDENT, sheet_name="Variable_Identity")
domain_meta  = pd.read_excel(DOMAIN_META, sheet_name="Domains")

print(f"BC            : {len(bc):>6}")
print(f"DSS           : {len(dss):>6}")
print(f"Variables     : {len(variables):>6}")
print(f"Var identity  : {len(var_ident):>6}")
print(f"Domain meta   : {len(domain_meta):>6}")

## Natural-name substitution (two-tier rule)

Identical to notebook 60 — direct hit first, then compositional
fallback via root subset after stripping the two-char domain prefix.

In [ ]:
_direct_tbl = (var_ident[var_ident['Subset'] != 'Root']
               .drop_duplicates(subset=['Variable'], keep='first')
               .set_index('Variable'))
_root_tbl   = (var_ident[var_ident['Subset'] == 'Root']
               .drop_duplicates(subset=['Variable'], keep='first')
               .set_index('Variable'))

def var_nn(variable):
    if variable in _direct_tbl.index:
        return _direct_tbl.loc[variable]['Natural_Name'], 'direct'
    if len(variable) > 2 and variable[2:] in _root_tbl.index:
        return _root_tbl.loc[variable[2:]]['Natural_Name'], 'compositional'
    return variable, 'unresolved'

def structural_type(domain_code):
    row = domain_meta[domain_meta['Domain'] == domain_code]
    if not len(row):
        return 'unknown'
    r = row.iloc[0]
    if r['Observation_Class'] != 'Findings':
        return 'non-findings'
    if r['Specimen_Based'] == 'Yes':
        return 'specimen-based-findings'
    if r['Measurement'] == 'Yes':
        return 'measurement-findings'
    return 'instrument-findings'

## Paragraph assembler

The Tier 2b paragraph is bands 1 + 3a of the template catalogue,
collapsed into prose. Shape:

1. **Opener sentence** — DSS name, `ds_id`, BC name, `bc_id`, domain,
   structural-type framing.
2. **Pins sentence** — what the DSS pins (variable → value pairs,
   with natural-name substitution).
3. **Open slots sentence** — what it leaves open, with value-list or
   codelist constraint where present.
4. **SDTMIG context sentence** — version where available; skipped
   otherwise.

Conditional construction — slots not populated are silently omitted.
No fabrication: if a slot has neither pinned value nor open list,
it appears as "open" with no constraint text.

In [ ]:
def classify_variables_for_dss(ds_id):
    """Return (pinned, open) lists of dicts describing this DSS's variables."""
    v = variables[variables['ds_id'] == ds_id]
    pinned, open_slots = [], []
    for _, r in v.iterrows():
        var = r['variable_name']
        if pd.isna(var):
            continue
        nn, src = var_nn(var)
        if pd.notna(r.get('assigned_term_value')):
            pinned.append({
                'variable': var, 'natural_name': nn, 'source': src,
                'role': r['role'],
                'value': r['assigned_term_value'],
                'codelist': r.get('codelist_submission_value')
                            if pd.notna(r.get('codelist_submission_value')) else None,
            })
        else:
            open_slots.append({
                'variable': var, 'natural_name': nn, 'source': src,
                'role': r['role'],
                'value_list': r['value_list'] if pd.notna(r.get('value_list')) else None,
                'codelist':   r['codelist_submission_value']
                              if pd.notna(r.get('codelist_submission_value')) else None,
            })
    return pinned, open_slots


def assemble_paragraph(ds_row, bc_row):
    ds_id   = ds_row['ds_id']
    dname   = ds_row['ds_short_name']
    bc_id   = ds_row['bc_id']
    bname   = bc_row['bc_short_name']
    domain  = ds_row['domain']
    stype   = structural_type(domain)
    since   = ds_row.get('sdtmig_start_version')

    pinned, open_slots = classify_variables_for_dss(ds_id)

    # Sentence 1 — opener
    s1 = (f"The {dname} specialisation ({ds_id}) realises the "
          f"{bname} biomedical concept ({bc_id}) as a row template "
          f"in SDTM domain {domain}")
    if stype not in ('unknown', 'non-findings'):
        s1 += f" — a {stype.replace('-', ' ')} pattern"
    s1 += "."

    # Sentence 2 — pins
    if pinned:
        parts = []
        for p in pinned:
            parts.append(f"{p['natural_name']} ({p['variable']}) = {p['value']}")
        s2 = "It pins " + "; ".join(parts) + "."
    else:
        s2 = "It pins no variables to fixed values at row-template scope."

    # Sentence 3 — open slots
    if open_slots:
        parts = []
        for o in open_slots:
            desc = f"{o['natural_name']} ({o['variable']})"
            if o['value_list']:
                desc += f" constrained to [{o['value_list']}]"
            elif o['codelist']:
                desc += f" drawn from codelist {o['codelist']}"
            parts.append(desc)
        s3 = "It leaves open the following slots: " + "; ".join(parts) + "."
    else:
        s3 = "No open slots are declared at row-template scope."

    # Sentence 4 — SDTMIG context
    if pd.notna(since) and str(since).strip():
        s4 = f"Added to SDTMIG in {since}."
    else:
        s4 = None

    parts = [s1, s2, s3]
    if s4:
        parts.append(s4)
    return " ".join(parts)


# Smoke on a couple of known DSSs
for probe in ['GLUCBLD', 'GLUCPL', 'XRAY', 'XRAYCHEST']:
    ds_row = dss[dss['ds_id']==probe]
    if not len(ds_row):
        print(f"{probe}: NOT FOUND")
        continue
    ds_row = ds_row.iloc[0]
    bc_row = bc[bc['bc_id']==ds_row['bc_id']].iloc[0]
    para = assemble_paragraph(ds_row, bc_row)
    print(f"\n--- {probe} ---")
    print(para)

## Iterate all 1,326 DSSs

In [ ]:
rows = []
bc_idx = bc.set_index('bc_id')

for _, ds_row in dss.iterrows():
    bc_id = ds_row['bc_id']
    if bc_id not in bc_idx.index:
        # DSS references a BC not in the BC sheet — flag and skip paragraph body
        rows.append({
            'ds_id'          : ds_row['ds_id'],
            'bc_id'          : bc_id,
            'ds_short_name'  : ds_row['ds_short_name'],
            'bc_short_name'  : None,
            'domain'         : ds_row['domain'],
            'structural_type': structural_type(ds_row['domain']),
            'n_pinned'       : None,
            'n_open'         : None,
            'has_unresolved' : None,
            'VLM_Narrative'  : f"[DANGLING-BC] DSS {ds_row['ds_id']} references bc_id={bc_id} not present in BC sheet.",
        })
        continue

    bc_row = bc_idx.loc[bc_id]
    if isinstance(bc_row, pd.DataFrame):
        bc_row = bc_row.iloc[0]

    pinned, open_slots = classify_variables_for_dss(ds_row['ds_id'])
    any_unresolved = any(p['source'] == 'unresolved' for p in pinned) or \
                     any(o['source'] == 'unresolved' for o in open_slots)

    rows.append({
        'ds_id'          : ds_row['ds_id'],
        'bc_id'          : bc_id,
        'ds_short_name'  : ds_row['ds_short_name'],
        'bc_short_name'  : bc_row['bc_short_name'],
        'domain'         : ds_row['domain'],
        'structural_type': structural_type(ds_row['domain']),
        'n_pinned'       : len(pinned),
        'n_open'         : len(open_slots),
        'has_unresolved' : any_unresolved,
        'VLM_Narrative'  : assemble_paragraph(ds_row, bc_row),
    })

narratives = pd.DataFrame(rows)
print(f"Assembled {len(narratives)} paragraphs over {narratives['domain'].nunique()} domains.")
print(narratives.groupby('structural_type').size())

## Coverage summary before write

- Paragraph row per DSS.
- Identity-resolution coverage — count of DSSs with any unresolved
  variable code in their paragraph.
- Dangling-BC flag count.

In [ ]:
print(f"Total DSSs       : {len(narratives)}")
print(f"Dangling BC      : {(narratives['VLM_Narrative'].astype(str).str.startswith('[DANGLING-BC]')).sum()}")
print(f"Any unresolved   : {narratives['has_unresolved'].sum()}")
print(f"Avg paragraph len: {narratives['VLM_Narrative'].str.len().mean():.0f} chars")
print(f"Max paragraph len: {narratives['VLM_Narrative'].str.len().max()} chars")

print("\nPer structural type:")
print(narratives.groupby('structural_type').agg(
    n_dss=('ds_id','count'),
    avg_pinned=('n_pinned','mean'),
    avg_open=('n_open','mean'),
    any_unresolved=('has_unresolved','sum'),
).round(1))

## Write `COSMoS_Graph_Narrative.xlsx`

Two sheets — README plus Narratives. Header colour on Narratives
uses **yellow** per the cdisc-for-ai convention for COSMoS-sourced
output. The file lives in `interim/` because it is exploratory; §5
decides whether it promotes into `COSMoS_Graph.xlsx` at close-out.

In [ ]:
YELLOW = PatternFill(start_color='FFE699', end_color='FFE699', fill_type='solid')
GREY   = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')
BOLD   = Font(bold=True)

wb = Workbook()

# README sheet
readme = wb.active
readme.title = 'README'
readme_lines = [
    ('COSMoS_Graph_Narrative.xlsx', None),
    ('', None),
    ('Purpose',
     'Tier 2b narrative layer over the COSMoS graph: one deterministically assembled '
     'English paragraph per DSS. Produced by '
     'sdtm-narrative/notebooks/40_assemble_narrative.ipynb.'),
    ('Grain', 'One row per DSS ({} DSSs total).'.format(len(narratives))),
    ('Source',
     'Assembled from cosmos-graph/interim/COSMoS_Graph.xlsx (BC, DSS, Variables) '
     'and sdtm-narrative/reference/SDTM_Variable_Identity.xlsx. No LLM at write time.'),
    ('Paragraph shape',
     'Four-sentence skeleton: (1) opener naming DSS/BC/domain/structural-type; '
     '(2) pins — what variables the DSS fixes to values; (3) open slots — '
     'variables constrained by value_list or codelist but not pinned; '
     '(4) SDTMIG start version where available.'),
    ('Natural-name substitution',
     'Two-tier rule: direct hit against SDTM_Variable_Identity first, '
     'compositional fallback (strip 2-char domain prefix, look up --REMAINDER '
     'against Root subset) otherwise. Unresolved codes surface in the '
     'has_unresolved flag.'),
    ('Template catalogue',
     'Paragraph corresponds to bands 1 + 3a of sdtm-narrative/reference/templates/ '
     'entries 01 and 02.'),
    ('', None),
    ('Columns', None),
    ('  ds_id', 'DSS code (unique primary key).'),
    ('  bc_id', 'Parent BC code.'),
    ('  ds_short_name', 'DSS short name from COSMoS_Graph.xlsx/DSS.'),
    ('  bc_short_name', 'BC short name from COSMoS_Graph.xlsx/BC.'),
    ('  domain', 'SDTM domain (two letters).'),
    ('  structural_type',
     'Derived from SDTM_Domain_Metadata: specimen-based-findings | '
     'measurement-findings | instrument-findings | non-findings | unknown.'),
    ('  n_pinned',
     'Number of Variables rows for this DSS with a non-null assigned_term_value.'),
    ('  n_open',
     'Number of Variables rows for this DSS with no assigned_term_value '
     '(whether or not value_list / codelist is present).'),
    ('  has_unresolved',
     'True if any variable in the paragraph failed the two-tier identity lookup.'),
    ('  VLM_Narrative', 'The Tier 2b paragraph.'),
    ('', None),
    ('Close-out decisions (§5 of COSMoS_Narrative_Layer.md)', None),
    ('  Promotion',
     'Open: whether this sheet folds back into COSMoS_Graph.xlsx as a '
     'Narratives sheet, stays here, or promotes separately. Decided at 3f.'),
    ('Header colour convention', 'Yellow (#FFE699) = COSMoS-sourced output.'),
]
for i, (k, v) in enumerate(readme_lines, start=1):
    readme.cell(row=i, column=1, value=k).font = BOLD if v is not None and k else Font(bold=False)
    if k and not v and i in (1,):
        readme.cell(row=i, column=1).font = Font(bold=True, size=14)
    if v is not None:
        c = readme.cell(row=i, column=2, value=v)
        c.alignment = Alignment(wrap_text=True, vertical='top')
readme.column_dimensions['A'].width = 34
readme.column_dimensions['B'].width = 110

# Narratives sheet
ws = wb.create_sheet('Narratives')
cols = ['ds_id','bc_id','ds_short_name','bc_short_name','domain',
        'structural_type','n_pinned','n_open','has_unresolved','VLM_Narrative']
for j, c in enumerate(cols, start=1):
    cell = ws.cell(row=1, column=j, value=c)
    cell.font = BOLD
    cell.fill = YELLOW
    cell.alignment = Alignment(wrap_text=False, vertical='top')

for i, r in enumerate(narratives.itertuples(index=False), start=2):
    row = r._asdict()
    for j, c in enumerate(cols, start=1):
        val = row[c]
        if pd.isna(val):
            val = None
        cell = ws.cell(row=i, column=j, value=val)
        if c == 'VLM_Narrative':
            cell.alignment = Alignment(wrap_text=True, vertical='top')

# Column widths
widths = {'ds_id':16,'bc_id':10,'ds_short_name':36,'bc_short_name':36,'domain':8,
          'structural_type':22,'n_pinned':9,'n_open':9,'has_unresolved':14,
          'VLM_Narrative':120}
for j, c in enumerate(cols, start=1):
    ws.column_dimensions[get_column_letter(j)].width = widths[c]

ws.freeze_panes = 'A2'

wb.save(OUT_XLSX)
print(f"wrote {OUT_XLSX}  ({OUT_XLSX.stat().st_size:,} bytes)")

## Smoke validation

- Row count matches DSS sheet.
- Unresolved-variable-code inventory (distinct codes that failed the
  two-tier lookup, with DSS count).
- Five sample paragraphs spanning structural types.

In [ ]:
print(f"Row count check  : narratives={len(narratives)}  DSS={len(dss)}  "
      f"{'OK' if len(narratives)==len(dss) else 'MISMATCH'}")

# Inventory of unresolved variable codes
unres = {}
for _, ds_row in dss.iterrows():
    v = variables[variables['ds_id']==ds_row['ds_id']]
    for _, r in v.iterrows():
        var = r['variable_name']
        if pd.isna(var):
            continue
        if var_nn(var)[1] == 'unresolved':
            unres.setdefault(var, set()).add(ds_row['ds_id'])

if unres:
    print(f"\nUnresolved variable codes: {len(unres)}")
    for v, dss_set in sorted(unres.items(), key=lambda kv: -len(kv[1]))[:30]:
        print(f"  {v:12s}  used in {len(dss_set):>4} DSS(s)")
else:
    print("\nNo unresolved variable codes — identity coverage is complete.")

# Sample paragraphs across structural types
print("\nSamples:")
for stype in narratives['structural_type'].unique():
    sample = narratives[narratives['structural_type']==stype].iloc[0]
    print(f"\n[{stype}] {sample['ds_id']}")
    print(sample['VLM_Narrative'])

## Notes for 3e / 3f

- **Substitution coverage.** Any DSSs in `has_unresolved == True`
  cite a variable code the two-tier rule did not cover. These are
  expected to be zero if the Variable_Identity table is complete
  for the variable vocabulary the graph uses; any non-zero count is
  a gap to close either in identity building (3a) or in a
  compositional-fallback edge case (e.g. 3-char prefixes, suffix-only
  variables).
- **Deterministic round-trip (3e).** Every claim in every paragraph
  traces back to: the DSS row (for ds_id / ds_short_name / domain /
  sdtmig_start_version), the BC row (for bc_id / bc_short_name),
  and the Variables rows for this ds_id (for pins / open slots).
  No field in the paragraph comes from anywhere else.
- **Promotion (§5).** The current output lives at
  `sdtm-narrative/interim/COSMoS_Graph_Narrative.xlsx`. Promotion
  into `cosmos-graph/interim/COSMoS_Graph.xlsx` as a `Narratives`
  sheet is the close-out candidate; decision pending 3e validation.